# 🛠️ Defining Tools in LangChain

## Learning Objectives
In this notebook, you will learn:
1. **What a tool is** - A name, a description and an argument schema wrapped around a function
2. **Built-in tools** - Using pre-built tools like Wikipedia and Tavily Search, and customizing them
3. **Custom tools** - Creating your own tools with the `@tool` decorator
4. **Type-safe tools** - Validating inputs with a Pydantic schema
5. **What the model sees** - The JSON schema a tool turns into when it is handed to an LLM

No LLM is called in this notebook: it is only about *defining* tools. Handing them to a model and running the calls is the next notebook, `02_tool_calling_manual_loop`.

## Prerequisites
- Familiarity with Python decorators and type hints
- Dependencies installed with `uv sync` at the repository root
- In the project-root `.env`: `TAVILY_API_KEY` (Tavily Search, [get a free key](https://tavily.com/#api)) and, optionally, `WEATHER_API_KEY` (OpenWeatherMap, [get a free key](https://openweathermap.org/api)) for the weather tool

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env` and keep the output free of library warnings.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
from warnings import filterwarnings

from dotenv import find_dotenv, load_dotenv

filterwarnings('ignore')

# Reads TAVILY_API_KEY and WEATHER_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

✅ Environment ready


---
## 🔧 Part 1: Exploring Built-in Tools

LangChain provides several pre-built tools that you can use out-of-the-box. These tools wrap common APIs and services, making it easy to give your LLM access to external capabilities.

### Key Concepts:
- **Tool**: An interface that an agent/LLM can use to interact with the world
- **API Wrapper**: Handles the low-level API calls
- **Tool Attributes**: Each tool has `name`, `description`, and `args`

### 1.1 📚 Wikipedia Tool

The Wikipedia tool enables you to tap into Wikipedia's vast knowledge base through their API. This is useful for retrieving factual information about entities, concepts, and events.

In [2]:
# ============================================================================
# WIKIPEDIA TOOL: Setup and Configuration
# ============================================================================
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Configure the Wikipedia API wrapper
# - top_k_results: Number of Wikipedia pages to return (default: 3)
# - doc_content_chars_max: Maximum characters to return per document (default: 4000)
wiki_api_wrapper = WikipediaAPIWrapper(
    top_k_results=3,
    doc_content_chars_max=8000
)

# Create the Wikipedia tool by wrapping the API
wiki_tool = WikipediaQueryRun(
    api_wrapper=wiki_api_wrapper, 
    features="lxml"  # Use lxml parser for HTML parsing
)

# Inspect the tool's attributes - these are what the LLM uses to understand the tool
print("📋 Tool Name:", wiki_tool.name)
print("📝 Tool Description:", wiki_tool.description)
print("📦 Tool Arguments:", wiki_tool.args)

📋 Tool Name: wikipedia
📝 Tool Description: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
📦 Tool Arguments: {'query': {'description': 'query to look up on wikipedia', 'title': 'Query', 'type': 'string'}}


In [3]:
# ============================================================================
# WIKIPEDIA TOOL: Demonstration
# ============================================================================
# Let's test the Wikipedia tool by searching for information about Microsoft

result = wiki_tool.invoke({"query": "Microsoft"})
print("🔍 Wikipedia Search Result for 'Microsoft':")
print("=" * 60)
print(result[:2000])  # Print first 2000 characters for readability
print("\n... [truncated for display]")

🔍 Wikipedia Search Result for 'Microsoft':
Page: Microsoft
Summary: Microsoft Corporation is an American multinational technology company headquartered in Redmond, Washington. The company became influential in the rise of personal computers through software like Windows and has since expanded into areas such as Internet services, cloud computing, artificial intelligence, video gaming, and more. A Big Tech company, Microsoft is the largest software company by revenue, one of the most valuable public companies, and one of the most valuable brands globally.
Founded in 1975 by Bill Gates and Paul Allen to market BASIC interpreters for the Altair 8800, Microsoft rose to dominate the PC operating system market with MS-DOS in the mid-1980s, followed by Windows. The company's 1986 initial public offering and the subsequent rise in its share price created three billionaires and an estimated 12,000 millionaires among Microsoft employees. Since the 1990s, it has increasingly diversified its busin

#### 🎨 Customizing Built-in Tools

You can customize any built-in tool by wrapping it with your own name, description, and behavior. This is useful when you want to:
- Provide a more specific description for your use case
- Rename the tool for clarity
- Add pre/post-processing logic

In [4]:
# ============================================================================
# CUSTOMIZING TOOLS: Creating a Custom Wikipedia Tool
# ============================================================================
from langchain_core.tools import Tool

# Create a custom version of the Wikipedia tool with our own name and description
wiki_tool_custom = Tool(
    name="Wikipedia",
    func=wiki_api_wrapper.run,  # Use the same underlying function
    description="Useful when you need a detailed answer about general knowledge, "
                "historical facts, famous people, places, or scientific concepts."
)

# Compare the custom tool attributes
print("📋 Custom Tool Name:", wiki_tool_custom.name)
print("📝 Custom Tool Description:", wiki_tool_custom.description)
print("📦 Custom Tool Arguments:", wiki_tool_custom.args)

# Test the custom tool (note: uses 'tool_input' instead of 'query')
print("\n🔍 Testing custom tool with 'AI':")
print(wiki_tool_custom.invoke({"tool_input": "AI"})[:500])

📋 Custom Tool Name: Wikipedia
📝 Custom Tool Description: Useful when you need a detailed answer about general knowledge, historical facts, famous people, places, or scientific concepts.
📦 Custom Tool Arguments: {'tool_input': {'type': 'string'}}

🔍 Testing custom tool with 'AI':


Page: Artificial intelligence
Summary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their cha


### 1.2 🔍 Tavily Search Tool

**Tavily Search API** is a search engine optimized for LLMs and RAG (Retrieval-Augmented Generation). It provides:
- Real-time web search results
- Clean, structured output
- Advanced search capabilities
- Raw content extraction from web pages

In [5]:
# ============================================================================
# TAVILY SEARCH TOOL: Setup and Configuration
# ============================================================================
# Note: Requires TAVILY_API_KEY in your environment variables

from langchain_community.tools.tavily_search import TavilySearchResults

# Configure the Tavily search tool
# - max_results: Maximum number of search results to return
# - search_depth: 'basic' or 'advanced' (advanced provides more detailed results)
# - include_raw_content: Whether to include the raw HTML content
tavily_tool = TavilySearchResults(
    max_results=5,
    search_depth='advanced',
    include_raw_content=True
)

# Inspect tool attributes
print("📋 Tool Name:", tavily_tool.name)
print("📝 Tool Description:", tavily_tool.description)
print("📦 Tool Arguments:", tavily_tool.args)

📋 Tool Name: tavily_search_results_json
📝 Tool Description: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.
📦 Tool Arguments: {'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


/var/folders/16/l0pt1ckj4hg1jzr4894_3f340000gn/T/ipykernel_20207/502350992.py:12: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(


In [6]:
# ============================================================================
# TAVILY SEARCH TOOL: Demonstration
# ============================================================================
# Let's search for information about Microsoft

results = tavily_tool.invoke("Tell me about Microsoft")

print("🔍 Tavily Search Results:")
print("=" * 60)
for i, result in enumerate(results[:3], 1):  # Show first 3 results
    print(f"\n📄 Result {i}:")
    print(f"   URL: {result.get('url', 'N/A')}")
    print(f"   Content: {result.get('content', 'N/A')[:200]}...")

🔍 Tavily Search Results:

📄 Result 1:
   URL: https://www.ebsco.com/research-starters/computer-science/microsoft
   Content: RESEARCH STARTER

# Microsoft

Microsoft Corporation is a leading American multinational technology company, renowned for its significant impact on the computer software industry. Founded in the mid-1...

📄 Result 2:
   URL: https://www.britannica.com/money/Microsoft-Corporation
   Content: Microsoft Corporation is an American multinational technology company that has been a key player in the computer industry since its founding in 1975. Headquartered in Redmond, Washington, Microsoft in...

📄 Result 3:
   URL: https://news.microsoft.com/facts-about-microsoft
   Content: # Facts About Microsoft

Microsoft creates platforms and tools powered by AI to deliver innovative solutions that meet the evolving needs of our customers. The technology company is committed to makin...


---
## 🔨 Part 2: Building Custom Tools

While built-in tools are useful, you'll often need to create custom tools for your specific use cases. LangChain provides several ways to create tools.

### Key Components of a Tool:
1. **Name** - A unique identifier for the tool
2. **Description** - Explains what the tool does (LLM uses this to decide when to use it)
3. **Args Schema** - JSON schema defining the input parameters
4. **Function** - The actual code that runs when the tool is invoked
5. **Return Direct** - Whether to return the result directly to the user

### 2.1 ➕ Building a Simple Math Tool

The simplest way to create a tool is using the `@tool` decorator. Let's create a basic multiplication tool.

In [7]:
# ============================================================================
# CUSTOM TOOLS: Simple Tool with @tool Decorator
# ============================================================================
from langchain_core.tools import tool

@tool
def multiply(a, b):
    """Multiply two numbers."""
    return a * b

# Inspect the automatically generated tool attributes
print("📋 Tool Name:", multiply.name)
print("📝 Tool Description:", multiply.description)
print("📦 Tool Arguments:", multiply.args)
print("🔧 Tool Type:", type(multiply))

# Test the tool with different inputs
print("\n🧮 Testing multiply tool:")
print(f"   2 × 3 = {multiply.invoke({'a': 2, 'b': 3})}")
print(f"   2.1 × 3.2 = {multiply.invoke({'a': 2.1, 'b': 3.2})}")

# Note: Without type hints, the tool accepts any type!
print(f"   2 × 'abc' = {multiply.invoke({'a': 2, 'b': 'abc'})} (string repeated!)")

📋 Tool Name: multiply
📝 Tool Description: Multiply two numbers.
📦 Tool Arguments: {'a': {'title': 'A'}, 'b': {'title': 'B'}}
🔧 Tool Type: <class 'langchain_core.tools.structured.StructuredTool'>

🧮 Testing multiply tool:
   2 × 3 = 6
   2.1 × 3.2 = 6.720000000000001
   2 × 'abc' = abcabc (string repeated!)


### 2.2 🔒 Building a Type-Safe Tool with Pydantic

The simple `@tool` decorator doesn't enforce type checking. For production use, you should use **Pydantic** schemas to validate inputs. This approach:
- Enforces type safety
- Provides better descriptions for each argument
- Generates proper JSON schemas for LLM understanding

In [8]:
# ============================================================================
# CUSTOM TOOLS: Type-Safe Tool with Pydantic Schema
# ============================================================================
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

# Define the input schema using Pydantic
# This provides type validation and rich descriptions
class CalculatorInput(BaseModel):
    """Input schema for calculator operations."""
    a: float = Field(description="The first number to multiply")
    b: float = Field(description="The second number to multiply")

# Define the function with proper type hints
def multiply_safe(a: float, b: float) -> float:
    """Multiply two numbers safely."""
    return a * b

# Create a StructuredTool with the Pydantic schema
multiply = StructuredTool.from_function(
    func=multiply_safe,
    name="multiply",
    description="Use this tool to multiply two numbers together. "
                "Both inputs must be valid numbers (integers or decimals).",
    args_schema=CalculatorInput,
    return_direct=True  # Return result directly without further processing
)

# Inspect the tool - note the improved argument schema
print("📋 Tool Name:", multiply.name)
print("📝 Tool Description:", multiply.description)
print("📦 Tool Arguments:", multiply.args)

# Test with valid input
print("\n✅ Valid input (2 × 3):", multiply.invoke({"a": 2, "b": 3}))

# Test with invalid input - this will raise a validation error!
print("\n❌ Invalid input (2 × 'abc'):")
try:
    multiply.invoke({"a": 2, "b": 'abc'})
except Exception as e:
    print(f"   Validation Error: {type(e).__name__}")

📋 Tool Name: multiply
📝 Tool Description: Use this tool to multiply two numbers together. Both inputs must be valid numbers (integers or decimals).
📦 Tool Arguments: {'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'number'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'number'}}

✅ Valid input (2 × 3): 6.0

❌ Invalid input (2 × 'abc'):
   Validation Error: ValidationError


### 2.3 🌐 Building a Web Search & Information Extraction Tool

Let's create a more sophisticated tool that:
1. Searches the web using Tavily
2. Extracts content from the found URLs using MarkItDown
3. Returns clean, structured information

In [9]:
# ============================================================================
# CUSTOM TOOLS: Advanced Web Search & Content Extraction Tool
# ============================================================================
from markitdown import MarkItDown
from langchain_community.tools.tavily_search import TavilySearchResults
from tqdm import tqdm
import requests

# Initialize the search tool and markdown converter
tavily_tool = TavilySearchResults(
    max_results=5,
    search_depth='advanced',
    include_answer=False,
    include_raw_content=True
)
md = MarkItDown()

@tool
def search_web_extract_info(query: str) -> list:
    """
    Search the web for a query and extract useful information from the search results.
    
    This tool:
    1. Searches the web using Tavily Search API
    2. Visits each result URL
    3. Extracts and cleans the text content
    4. Returns a list of extracted documents
    
    Args:
        query: The search query to look up on the web
        
    Returns:
        list: A list of extracted text content from web pages
    """
    # Step 1: Search the web
    results = tavily_tool.invoke(query)
    docs = []
    
    # Step 2: Extract content from each URL
    for result in tqdm(results, desc="Extracting content"):
        try:
            extracted_info = md.convert(result['url'])
            text_title = extracted_info.title.strip()
            text_content = extracted_info.text_content.strip()
            docs.append(text_title + '\n' + text_content)
        except Exception as e:
            print(f'⚠️ Extraction blocked for url: {result["url"]}')
            pass
    
    return docs

print("✅ Web search tool created successfully!")
print("📋 Tool Name:", search_web_extract_info.name)
print("📝 Tool Description:", search_web_extract_info.description[:200] + "...")
print("\n... [truncated for display]")

✅ Web search tool created successfully!
📋 Tool Name: search_web_extract_info
📝 Tool Description: Search the web for a query and extract useful information from the search results.

This tool:
1. Searches the web using Tavily Search API
2. Visits each result URL
3. Extracts and cleans the text con...

... [truncated for display]


In [10]:
# ============================================================================
# WEB SEARCH TOOL: Demonstration
# ============================================================================
# Test the web search tool (this may take a few seconds)

docs = search_web_extract_info.invoke('OpenAI GPT-4o')

print(f"\n📚 Extracted {len(docs)} documents")
if docs:
    print("\n📄 First document preview:")
    print("=" * 60)
    print(docs[0][:500] + "...")

Extracting content:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting content:  20%|██        | 1/5 [00:00<00:00,  4.74it/s]

⚠️ Extraction blocked for url: https://en.wikipedia.org/wiki/GPT-4o


Extracting content:  40%|████      | 2/5 [00:00<00:00,  3.30it/s]

⚠️ Extraction blocked for url: https://openrouter.ai/openai/gpt-4o


Extracting content:  60%|██████    | 3/5 [00:02<00:01,  1.19it/s]

Extracting content:  80%|████████  | 4/5 [00:02<00:00,  1.59it/s]

Extracting content: 100%|██████████| 5/5 [00:04<00:00,  1.04s/it]

Extracting content: 100%|██████████| 5/5 [00:04<00:00,  1.21it/s]


📚 Extracted 3 documents

📄 First document preview:
GPT-4o 101: What It Is and How It Works | Grammarly
[Grammarly Home](https://www.grammarly.com)

* Product

  + Learn
    - [Features](https://www.grammarly.com/features)
    - [AI agents](https://www.grammarly.com/ai-agents)
    - [AI assistant](https://www.grammarly.com/go-ai-assistant)
    - [AI at Grammarly](https://www.grammarly.com/ai)
    - [Trust & security](https://www.grammarly.com/trust)
    - [Superhuman](https://superhuman.com)
  + Use Grammarly
    - [Docs](https://www.grammarly.co...


### 2.4 🌤️ Building a Weather Tool

Let's create a tool that fetches real-time weather data using the OpenWeatherMap API.

In [11]:
# ============================================================================
# CUSTOM TOOLS: Weather API Tool
# ============================================================================
import requests
import rich

# Get the Weather API key from environment
WEATHER_API_KEY = os.getenv('WEATHER_API_KEY')

@tool
def get_weather(query: str) -> dict:
    """
    Get the current weather for a city using OpenWeatherMap API.
    
    Args:
        query: The name of the city to get weather for (e.g., 'Bangalore', 'Mumbai')
        
    Returns:
        dict: Weather data including temperature, humidity, and conditions,
              or an error message if the city is not found
    """
    url = f"https://api.openweathermap.org/data/2.5/weather?q={query},IN&appid={WEATHER_API_KEY}&units=metric"
    
    response = requests.get(url)
    data = response.json()
    
    if data.get("name"):
        return data
    else:
        return {"error": "Weather Data Not Found", "city": query}

# Test the weather tool
print("🌤️ Testing Weather Tool:")
result = get_weather.invoke("Bangalore")
rich.print_json(data=result)

🌤️ Testing Weather Tool:


{
  "coord": {
    "lon": 77.6033,
    "lat": 12.9762
  },
  "weather": [
    {
      "id": 500,
      "main": "Rain",
      "description": "light rain",
      "icon": "10d"
    }
  ],
  "base": "stations",
  "main": {
    "temp": 27.88,
    "feels_like": 30.82,
    "temp_min": 27.88,
    "temp_max": 27.88,
    "pressure": 1013,
    "humidity": 73,
    "sea_level": 1013,
    "grnd_level": 915
  },
  "visibility": 10000,
  "wind": {
    "speed": 2.66,
    "deg": 17,
    "gust": 4.41
  },
  "rain": {
    "1h": 0.49
  },
  "clouds": {
    "all": 98
  },
  "dt": 1790771758,
  "sys": {
    "country": "IN",
    "sunrise": 1790728723,
    "sunset": 1790772035
  },
  "timezone": 19800,
  "id": 1277333,
  "name": "Bengaluru",
  "cod": 200
}

---
## 🔍 Part 3: What the Model Actually Sees

A tool never reaches the model as Python code. When you hand tools to an LLM, LangChain converts each one into a **JSON schema**: the `name`, the `description` (your docstring) and a typed `parameters` object. That schema is the *only* thing the model reads when it decides whether to call a tool and with what arguments.

> **Key Insight**: The quality of your docstring and argument descriptions directly affects how well the model selects and uses the tool. A vague description produces wrong or missing calls, however good the code behind it is.

In [12]:
# ============================================================================
# THE MODEL'S VIEW: Convert tools to the JSON schema sent to the LLM
# ============================================================================
import json

from langchain_core.utils.function_calling import convert_to_openai_tool

# The Pydantic-typed multiply from 2.2 and the weather tool from 2.4
for t in [multiply, get_weather]:
    print(f"📋 {t.name}")
    print(json.dumps(convert_to_openai_tool(t), indent=2))
    print("-" * 60)

📋 multiply
{
  "type": "function",
  "function": {
    "name": "multiply",
    "description": "Use this tool to multiply two numbers together. Both inputs must be valid numbers (integers or decimals).",
    "parameters": {
      "properties": {
        "a": {
          "description": "The first number to multiply",
          "type": "number"
        },
        "b": {
          "description": "The second number to multiply",
          "type": "number"
        }
      },
      "required": [
        "a",
        "b"
      ],
      "type": "object"
    }
  }
}
------------------------------------------------------------
📋 get_weather
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get the current weather for a city using OpenWeatherMap API.\n\nArgs:\n    query: The name of the city to get weather for (e.g., 'Bangalore', 'Mumbai')\n\nReturns:\n    dict: Weather data including temperature, humidity, and conditions,\n          or an error message if the 

Compare the two schemas. The Pydantic-backed `multiply` gives each argument a type (`number`) and a description, so the model knows exactly what to pass. The weather tool only says `query` is a string: its docstring has to carry the rest.

---
## 📝 Summary

In this notebook, we learned:

### 1. Built-in Tools
- **WikipediaQueryRun**: Query Wikipedia for information
- **TavilySearchResults**: Web search optimized for LLMs
- Built-in tools can be re-wrapped with your own name and description

### 2. Custom Tools
- Use the `@tool` decorator for simple tools; the docstring becomes the description
- Use `StructuredTool` with a Pydantic schema for type-safe tools that reject bad input

### 3. What the Model Sees
- Every tool becomes a JSON schema (`name`, `description`, `parameters`)
- Descriptions are the interface: the model chooses tools by reading them

### Next Steps
- **`02_tool_calling_manual_loop`**: bind tools to an LLM, read the tool calls it emits, and run the execution loop yourself